# Grounding accuracy eval only -- no training

**Why this notebook exists**: `kaggle_finetune_grounding_dino_v3_dota.ipynb`'s training actually
*succeeded* (10h17m, real checkpoints saved, sane periodic COCO metrics) -- it only errored
afterward, in its own Section 9 eval cell, on a `ModuleNotFoundError` caused by an upstream
Open-GroundingDino repo restructure (verified live: `groundingdino.util.inference` imports
`groundingdino.datasets`/`groundingdino.models`, neither of which exist anymore -- `datasets/`,
`models/`, `tools/`, `config/` all moved to the repo root at some point). That's fixed in v3's own
notebook file now, for any future from-scratch run -- but re-pushing the *whole* v3 notebook here
would burn another 10+ hours re-training a checkpoint that already exists, which this account's
GPU quota (4.4 of 30 weekly hours left at the time of writing) can't afford.

This notebook does only what actually needs the checkpoint: load it (already trained, uploaded as
a Kaggle input -- see the markdown cell below) and run the same Acc@0.5/Acc@0.7/mIoU protocol
every other version of this eval used, three ways (zero-shot / current v1 checkpoint / this new
one), so the real numbers exist without spending training-scale GPU time to get them.

Run cells top to bottom. Kaggle: enable **Internet** and a **GPU** in the notebook's Settings
panel before starting.

In [ ]:
import torch, subprocess
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" -", torch.cuda.get_device_name(i))

## 0. GPU compatibility check

Same as every other notebook in this project -- this session's preinstalled PyTorch only supports
sm_70+, which silently drops the Pascal-generation P100 (sm_60) Kaggle can still hand out.

In [ ]:
import subprocess, sys

try:
    cc_raw = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
    ).strip().splitlines()[0]
    major, minor = cc_raw.split(".")
    needed_sm = f"sm_{major}{minor}"
except Exception as e:
    needed_sm = None
    print(f"Could not query GPU compute capability via nvidia-smi ({e}) -- skipping the compatibility check.")

if needed_sm:
    check = subprocess.run(
        [sys.executable, "-c",
         "import torch; print(' '.join(torch.cuda.get_arch_list()) if torch.cuda.is_available() else '')"],
        capture_output=True, text=True,
    )
    supported = check.stdout.split()
    if needed_sm not in supported:
        print(f"GPU needs {needed_sm}, not in the preinstalled torch build's supported list "
              f"{supported} -- reinstalling a CUDA 11.8 build (covers Pascal through Hopper)...")
        subprocess.run(["pip", "uninstall", "-y", "-q", "torch", "torchvision", "torchaudio"], check=True)
        subprocess.run(
            ["pip", "install", "-q", "torch", "torchvision", "torchaudio",
             "--index-url", "https://download.pytorch.org/whl/cu118"],
            check=True,
        )
        print("Reinstalled -- the check above ran in a subprocess, so the next cell's `import torch` is fresh.")
    else:
        print(f"GPU compute capability {needed_sm} already supported by the preinstalled build.")

## 1. Setup — clone Open-GroundingDino, install deps, build the CUDA ops

Needed even for eval-only: `build_model()` constructs the same architecture, which needs the
compiled deformable-attention op.

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/longzw1997/Open-GroundingDino.git
%cd Open-GroundingDino
!pip install -q -r requirements.txt
!pip install -q "transformers<5"
%cd /kaggle/working/Open-GroundingDino/models/GroundingDINO/ops
!python setup.py build install
!python test.py
%cd /kaggle/working/Open-GroundingDino

## 2. Download the pretrained Swin-T weights (for the zero-shot baseline)

In [ ]:
%cd /kaggle/working/Open-GroundingDino
!mkdir -p weights
!wget -q -P weights https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
print("Downloaded:", __import__("os").path.getsize("weights/groundingdino_swint_ogc.pth"), "bytes")

from transformers import AutoTokenizer, AutoModel
AutoTokenizer.from_pretrained("bert-base-uncased")
AutoModel.from_pretrained("bert-base-uncased")
print("BERT cached.")

## 2b. Locate both checkpoints

Both checkpoints are mounted as regular dataset inputs, same as every other notebook in this
project: the current one (`dior-rsvg-finetuned-v1`) and the new one
(`dior-rsvg-finetuned-v3`, uploaded from this run's `checkpoint_best_regular.pth`), both listed
in `dataset_sources` in this kernel's metadata. An earlier attempt tried referencing the
training kernel's own output directly via `kernel_sources` instead, to skip the download +
re-upload round trip -- the Kaggle API rejected it ("not valid kernel sources"), so this reverts
to the same manual-upload pattern every other checkpoint in this project already uses.


In [ ]:
import glob, os

def find_checkpoint(filename_glob):
    matches = glob.glob(f"/kaggle/input/**/{filename_glob}", recursive=True)
    if not matches:
        print("Contents of /kaggle/input:")
        for root, dirs, files in os.walk("/kaggle/input"):
            for f in files:
                print(" ", os.path.join(root, f))
    assert matches, f"No checkpoint found matching {filename_glob} under /kaggle/input -- see the markdown cell above."
    return matches[0]

CURRENT_CKPT = find_checkpoint("dior_rsvg_finetuned.pth")
NEW_CKPT = find_checkpoint("checkpoint_best_regular.pth")
print("CURRENT_CKPT =", CURRENT_CKPT)
print("NEW_CKPT =", NEW_CKPT)

## 3. Download DIOR-RSVG (only need the test split + images)

In [ ]:
%cd /kaggle/working
!gdown --folder "https://drive.google.com/drive/folders/1hTqtYsC6B-m4ED2ewx5oKuYZV13EoJp_" -O DIOR_RSVG
!echo "---"
!find DIOR_RSVG -maxdepth 2 | head -20

import glob, os
for zip_path in glob.glob("DIOR_RSVG/*.zip"):
    print(f"Extracting and removing {zip_path} ...")
    !unzip -q -o {zip_path} -d DIOR_RSVG
    os.remove(zip_path)
!du -sh DIOR_RSVG

## 4. Parse the XML annotations, keep just the test split

Same parser as the training notebooks (mirrors the official `data_loader.py`, verified against the
actual upstream source) -- only `test_records` actually gets used below.

In [ ]:
import os, json, pickle
import xml.etree.ElementTree as ET
from PIL import Image

DIOR_ROOT = "/kaggle/working/DIOR_RSVG"
ANNO_DIR  = os.path.join(DIOR_ROOT, "Annotations")
IMG_DIR   = os.path.join(DIOR_ROOT, "JPEGImages")

def load_split_ids(split):
    with open(os.path.join(DIOR_ROOT, f"{split}.txt")) as f:
        return set(int(x.strip()) for x in f if x.strip())

def get_image_size(xml_root, image_path):
    w_el, h_el = xml_root.find("./size/width"), xml_root.find("./size/height")
    if w_el is not None and h_el is not None:
        return int(w_el.text), int(h_el.text)
    with Image.open(image_path) as im:
        return im.size

def parse_all_objects():
    xml_files = sorted(
        os.path.join(dp, f) for dp, _, fs in os.walk(ANNO_DIR) for f in fs if f.endswith(".xml")
    )
    records, count = [], 0
    for xp in xml_files:
        root = ET.parse(xp).getroot()
        filename = root.find("./filename").text
        w, h = get_image_size(root, os.path.join(IMG_DIR, filename))
        for member in root.findall("object"):
            category = member[0].text
            x1, y1, x2, y2 = (float(member[2][0].text), float(member[2][1].text),
                               float(member[2][2].text), float(member[2][3].text))
            expression = member[3].text
            records.append(dict(index=count, filename=filename, category=category,
                                 bbox=[x1, y1, x2, y2], width=w, height=h, expression=expression))
            count += 1
    return records

records = parse_all_objects()
test_ids = load_split_ids("test")
test_records = [r for r in records if r["index"] in test_ids]
print(f"test={len(test_records)} (paper: 7500)")

## 5. Grounding accuracy: zero-shot vs. current checkpoint vs. this run's new one

Same reimplemented load_model/load_image/predict as the fixed v3 notebook -- `groundingdino.util.inference`
imports `groundingdino.datasets`/`groundingdino.models`, neither of which exist in the current
Open-GroundingDino repo layout (verified live -- see v3 notebook's own note on this). Built
directly against the paths `main.py` itself successfully uses for training instead.

In [ ]:
import sys, os
sys.path.insert(0, "/kaggle/working/Open-GroundingDino")
import torch
from PIL import Image

from util.slconfig import SLConfig
from models.GroundingDINO import build_groundingdino as build_model  # models.build_model itself calls an undefined bare build() -- see notebook's own note
from groundingdino.util.utils import clean_state_dict, get_phrases_from_posmap
import datasets.transforms as T

def load_image(image_path):
    image_pil = Image.open(image_path).convert("RGB")
    transform = T.Compose([
        T.RandomResize([800], max_size=1333),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    image, _ = transform(image_pil, None)
    return image_pil, image

def load_model(model_config_path, model_checkpoint_path, device="cuda"):
    args = SLConfig.fromfile(model_config_path)
    args.device = device
    # cfg_odvg.py defaults to use_coco_eval=True, which makes build_groundingdino's internal
    # PostProcess try to load a real COCO annotation file via args.coco_val_path (only ever set
    # by main.py at training time from the dataset config, never a static config field) -- not
    # needed here since evaluate() never touches postprocessors, only raw model outputs, so this
    # is a placeholder to satisfy the constructor, not a real category list.
    args.use_coco_eval = False
    args.label_list = ["object"]
    model, _, _ = build_model(args)  # build_groundingdino returns (model, criterion, postprocessors)
    checkpoint = torch.load(model_checkpoint_path, map_location="cpu", weights_only=False)
    model.load_state_dict(clean_state_dict(checkpoint["model"]), strict=False)
    model.eval()
    return model.to(device)

def predict(model, image, caption, box_threshold=0.25, text_threshold=0.25, device="cuda"):
    caption = caption.lower().strip()
    if not caption.endswith("."):
        caption += "."
    image = image.to(device)
    with torch.no_grad():
        outputs = model(image[None], captions=[caption])
    logits = outputs["pred_logits"].sigmoid()[0]
    boxes = outputs["pred_boxes"][0]
    filt_mask = logits.max(dim=1)[0] > box_threshold
    logits_filt = logits[filt_mask].cpu()
    boxes_filt = boxes[filt_mask].cpu()
    tokenlizer = model.tokenizer
    tokenized = tokenlizer(caption)
    phrases = [get_phrases_from_posmap(logit > text_threshold, tokenized, tokenlizer) for logit in logits_filt]
    return boxes_filt, logits_filt.max(dim=1)[0], phrases

def box_cxcywh_to_xyxy_abs(box_norm, w, h):
    cx, cy, bw, bh = box_norm
    return [(cx - bw/2) * w, (cy - bh/2) * h, (cx + bw/2) * w, (cy + bh/2) * h]

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def evaluate(ckpt_path, cfg_path, test_records, image_root, box_th=0.25, text_th=0.25, limit=None):
    model = load_model(cfg_path, ckpt_path)
    ious = []
    subset = test_records[:limit] if limit else test_records
    for i, r in enumerate(subset):
        img_path = os.path.join(image_root, r["filename"])
        image_source, image = load_image(img_path)
        boxes, logits, phrases = predict(model=model, image=image, caption=r["expression"],
                                          box_threshold=box_th, text_threshold=text_th)
        if len(boxes) == 0:
            ious.append(0.0)
            continue
        best_idx = int(logits.argmax())
        pred_xyxy = box_cxcywh_to_xyxy_abs(boxes[best_idx].tolist(), r["width"], r["height"])
        ious.append(iou_xyxy(pred_xyxy, r["bbox"]))
        if i % 500 == 0:
            print(i, "/", len(subset))
    n = len(ious)
    acc5 = sum(x >= 0.5 for x in ious) / n
    acc7 = sum(x >= 0.7 for x in ious) / n
    miou = sum(ious) / n
    return {"n": n, "Acc@0.5": acc5, "Acc@0.7": acc7, "mIoU": miou}

image_root = "/kaggle/working/DIOR_RSVG/JPEGImages"
cfg_path = "/kaggle/working/Open-GroundingDino/config/cfg_odvg.py"  # the tools/ inference-only config lacks aux_loss/dn_labelbook_size/etc. that build_groundingdino requires -- see notebook note

print("Zero-shot baseline (pretrained, not fine-tuned) on a 1000-item subset:")
print(evaluate("/kaggle/working/Open-GroundingDino/weights/groundingdino_swint_ogc.pth",
                cfg_path, test_records, image_root, limit=1000))

In [ ]:
print("CURRENT checkpoint (v1, in the repo today) on the same 1000-item subset:")
print(evaluate(CURRENT_CKPT, cfg_path, test_records, image_root, limit=1000))

In [ ]:
print("NEW checkpoint (v3: DIOR-RSVG + VRSBench + DOTA) on the same 1000-item subset:")
print(evaluate(NEW_CKPT, cfg_path, test_records, image_root, limit=1000))

## Next

Compare the three numbers above. If v3 beats the current checkpoint, replace
`models/grounding/checkpoints/dior_rsvg_finetuned.pth` with the v3 checkpoint locally (same
`GroundingDINO_SwinT_OGC.py` config either way). No export/copy step here -- both checkpoints
being compared already exist as Kaggle inputs.